In [1]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib
import warnings
import statsmodels.api as sm

warnings.filterwarnings("ignore")

In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


### After separate cointegration tests across different pairs, GOOG VS CVX seemed to have strong cointegration, but good backtest results doesn't automatically prove an edge

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import vectorbt as vbt
import yfinance as yf

pairs = ["GOOG", "XOM"]
df3 = yf.download(
    pairs, start="2023-05-05", end="2025-05-05", progress=False
)  

stock_a = df3["Close"]["GOOG"]
stock_b = df3["Close"]["XOM"]

X = sm.add_constant(stock_b)
model = sm.OLS(stock_a, X).fit()
beta = model.params["XOM"]

window = 20
entry_threshold = 2.0
stop_loss_threshold = 3.5

spread = stock_a - (beta * stock_b)

rolling_mean = spread.rolling(window=window).mean()
rolling_std = spread.rolling(window=window).std()
zscore = (spread - rolling_mean) / rolling_std

long_entries = zscore < -entry_threshold
long_exits = (zscore.abs() <= 0.1) | (zscore.abs() >= stop_loss_threshold)

short_entries = zscore > entry_threshold
short_exits = (zscore.abs() <= 0.1) | (zscore.abs() >= stop_loss_threshold)


backtest = vbt.Portfolio.from_signals(
    stock_a,
    entries=long_entries,
    exits=long_exits,
    short_entries=short_entries,
    short_exits=short_exits,
    init_cash=10000.0,
    fees=0.001,
    freq="1D",
)

print(backtest.stats())
backtest.plot().show()

Start                         2023-05-05 00:00:00
End                           2025-05-02 00:00:00
Period                          500 days 00:00:00
Start Value                               10000.0
End Value                            13131.605752
Total Return [%]                        31.316058
Benchmark Return [%]                    56.847334
Max Gross Exposure [%]                      100.0
Total Fees Paid                         349.27461
Max Drawdown [%]                        15.515556
Max Drawdown Duration            95 days 00:00:00
Total Trades                                   15
Total Closed Trades                            15
Total Open Trades                               0
Open Trade PnL                                0.0
Win Rate [%]                            66.666667
Best Trade [%]                          10.130965
Worst Trade [%]                         -7.605736
Avg Winning Trade [%]                    4.953864
Avg Losing Trade [%]                    -4.035256


### The edge immediately vanishes with a different timeline

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import vectorbt as vbt
import yfinance as yf


pairs = ["GOOG", "XOM"]
df3 = yf.download(
    pairs, start="2023-05-05", end="2026-05-05", progress=False
)  

stock_a = df3["Close"]["GOOG"]
stock_b = df3["Close"]["XOM"]

X = sm.add_constant(stock_b)
model = sm.OLS(stock_a, X).fit()
beta = model.params["XOM"]

window = 20
entry_threshold = 2.0
stop_loss_threshold = 3.5

spread = stock_a - (beta * stock_b)

rolling_mean = spread.rolling(window=window).mean()
rolling_std = spread.rolling(window=window).std()
zscore = (spread - rolling_mean) / rolling_std

long_entries = zscore < -entry_threshold
long_exits = (zscore.abs() <= 0.1) | (zscore.abs() >= stop_loss_threshold)

short_entries = zscore > entry_threshold
short_exits = (zscore.abs() <= 0.1) | (zscore.abs() >= stop_loss_threshold)


backtest = vbt.Portfolio.from_signals(
    stock_a,
    entries=long_entries,
    exits=long_exits,
    short_entries=short_entries,
    short_exits=short_exits,
    init_cash=10000.0,
    fees=0.001,
    freq="1D",
)

print(backtest.stats())
backtest.plot().show()

Start                             2023-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                              751 days 00:00:00
Start Value                                   10000.0
End Value                                 4480.616707
Total Return [%]                           -55.193833
Benchmark Return [%]                       260.360453
Max Gross Exposure [%]                          100.0
Total Fees Paid                            393.466859
Max Drawdown [%]                            63.807213
Max Drawdown Duration               629 days 00:00:00
Total Trades                                       24
Total Closed Trades                                24
Total Open Trades                                   0
Open Trade PnL                                    0.0
Win Rate [%]                                45.833333
Best Trade [%]                              18.105657
Worst Trade [%]                            -31.331285
Avg Winning Trade [%]       

### Using a Kalman filter instead of OLS and making sure both the stocks are executed by vbt

In [8]:
import numpy as np
import pandas as pd
from pykalman import KalmanFilter
import vectorbt as vbt
import yfinance as yf


pairs = ["GOOG", "CVX"]
df3 = yf.download(pairs, start="2015-05-05", end="2025-05-05", progress=False)

try:
  stock_a = df3["Close"]["GOOG"]
  stock_b = df3["Close"]["CVX"]
except (KeyError, TypeError):
  stock_a = df3.xs("Close", level=0, axis=1)["GOOG"]
  stock_b = df3.xs("Close", level=0, axis=1)["CVX"]

price_data = pd.DataFrame({"GOOG": stock_a, "CVX": stock_b}).dropna()
s_a = price_data["GOOG"]
s_b = price_data["CVX"]


delta = 1e-3
trans_cov = delta / (1 - delta) * np.eye(2)
obs_mat = np.vstack([s_b.values, np.ones(len(s_b))]).T[:, np.newaxis, :]

kf = KalmanFilter(
    n_dim_obs=1,
    n_dim_state=2,
    initial_state_mean=[0, 0],
    initial_state_covariance=np.ones((2, 2)),
    transition_matrices=np.eye(2),
    observation_matrices=obs_mat,
    observation_covariance=1.0,
    transition_covariance=trans_cov,
)

state_means, _ = kf.filter(s_a.values)
beta = state_means[:, 0]
alpha = state_means[:, 1]

spread = s_a - (beta * s_b + alpha)
spread_series = pd.Series(spread, index=s_a.index)

window = 20  
rolling_mean = spread_series.rolling(window=window).mean()
rolling_std = spread_series.rolling(window=window).std()
zscore = (spread_series - rolling_mean) / rolling_std

entry_threshold = 2.5
stop_loss_threshold = 4.0


long_spread = zscore < -entry_threshold
short_spread = zscore > entry_threshold
exit_cond = (zscore.abs() <= 0.2) | (zscore.abs() >= stop_loss_threshold)

entries = pd.DataFrame({"GOOG": long_spread, "CVX": short_spread}, index=s_a.index)

short_entries = pd.DataFrame(
    {"GOOG": short_spread, "CVX": long_spread}, index=s_a.index
)

exits = pd.DataFrame({"GOOG": exit_cond, "CVX": exit_cond}, index=s_a.index)

short_exits = pd.DataFrame({"GOOG": exit_cond, "CVX": exit_cond}, index=s_a.index)


backtest = vbt.Portfolio.from_signals(
    close=price_data,
    entries=entries,
    exits=exits,
    short_entries=short_entries,
    short_exits=short_exits,
    init_cash=10000.0,
    fees=0.0002, 
    cash_sharing=True,
    group_by=True,
    freq="1D",
)

print(backtest.stats())
backtest.plot().show()

Start                            2015-05-05 00:00:00
End                              2025-05-02 00:00:00
Period                            2515 days 00:00:00
Start Value                                  10000.0
End Value                                11783.09274
Total Return [%]                           17.830927
Benchmark Return [%]                      312.241419
Max Gross Exposure [%]                    347.682471
Total Fees Paid                           292.773543
Max Drawdown [%]                           26.616589
Max Drawdown Duration             1181 days 00:00:00
Total Trades                                      52
Total Closed Trades                               52
Total Open Trades                                  0
Open Trade PnL                                   0.0
Win Rate [%]                               46.153846
Best Trade [%]                             30.653181
Worst Trade [%]                           -14.810141
Avg Winning Trade [%]                       3.